In [292]:
import pandas as pd
data_df = pd.read_csv('../data/challenge_224_data.csv')
attr_df = pd.read_csv('../data/challenge_224_attribute_info.csv')

In [293]:
data_df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [294]:
attr_df.head()

,Field1
0,"Attribute Information: (classes: edible=e, poi..."
1,"cap-shape: bell=b,conical=c,convex=x,flat=..."
2,"cap-surface: fibrous=f,grooves=g,scaly=y,s..."
3,"cap-color: brown=n,buff=b,cinnamon=c,gray=..."
4,"bruises: bruises=t,no=f"


In [295]:
# cleaning up the different formatting of the first entry

attr_df['Field1'] = attr_df['Field1'].str.replace('Attribute Information: (', '')
attr_df['Field1'] = attr_df['Field1'].str.replace(')', '')
attr_df['Field1'] = attr_df['Field1'].str.replace('classes', 'class')

In [296]:
# split strings to divide attributes and their assosciated key-value pairs

split_df = attr_df['Field1'].str.split(': ', n=1, expand=True)
attr_df['attribute'] = split_df[0].str.strip()
attr_df['key_value_pair'] = split_df[1]
attr_df.head()

,Field1,attribute,key_value_pair
0,"class: edible=e, poisonous=p",class,"edible=e, poisonous=p"
1,"cap-shape: bell=b,conical=c,convex=x,flat=...",cap-shape,"bell=b,conical=c,convex=x,flat=f, knobbed=k,su..."
2,"cap-surface: fibrous=f,grooves=g,scaly=y,s...",cap-surface,"fibrous=f,grooves=g,scaly=y,smooth=s"
3,"cap-color: brown=n,buff=b,cinnamon=c,gray=...",cap-color,"brown=n,buff=b,cinnamon=c,gray=g,green=r,pink=..."
4,"bruises: bruises=t,no=f",bruises,"bruises=t,no=f"


In [297]:
# put each key-value pair in its own row

attr_df['pairs_split'] = attr_df['key_value_pair'].str.split(',')
attr_df = attr_df.explode(['pairs_split'])

attr_df.head()

,Field1,attribute,key_value_pair,pairs_split
0,"class: edible=e, poisonous=p",class,"edible=e, poisonous=p",edible=e
0,"class: edible=e, poisonous=p",class,"edible=e, poisonous=p",poisonous=p
1,"cap-shape: bell=b,conical=c,convex=x,flat=...",cap-shape,"bell=b,conical=c,convex=x,flat=f, knobbed=k,su...",bell=b
1,"cap-shape: bell=b,conical=c,convex=x,flat=...",cap-shape,"bell=b,conical=c,convex=x,flat=f, knobbed=k,su...",conical=c
1,"cap-shape: bell=b,conical=c,convex=x,flat=...",cap-shape,"bell=b,conical=c,convex=x,flat=f, knobbed=k,su...",convex=x


In [298]:
# split keys and values into separate columns

split_df_2 = attr_df['pairs_split'].str.split('=', n=1, expand=True)
attr_df['value'] = split_df_2[0].str.strip()
attr_df['key'] = split_df_2[1].str.strip()
attr_df = attr_df[['attribute', 'value', 'key']]

attr_df.head()

,attribute,value,key
0,class,edible,e
0,class,poisonous,p
1,cap-shape,bell,b
1,cap-shape,conical,c
1,cap-shape,convex,x


In [299]:
# unpivot data to long format

data_df['record_id'] = data_df.index
data_df = pd.melt(data_df, id_vars='record_id')

data_df.head()

,record_id,variable,value
0,0,class,p
1,1,class,e
2,2,class,e
3,3,class,p
4,4,class,e


In [300]:
# join data with lookup table

df = pd.merge(attr_df, data_df, how='inner', left_on=['key', 'attribute'], right_on=['value', 'variable'])
df = df[['record_id', 'attribute', 'value_x']]

df.head()

,record_id,attribute,value_x
0,1,class,edible
1,2,class,edible
2,4,class,edible
3,5,class,edible
4,6,class,edible


In [301]:
df.pivot(index='record_id', columns=['attribute'], values=['value_x'])

value_x                                                             \
attribute  bruises cap-color cap-shape cap-surface      class gill-attachment   
record_id                                                                       
0          bruises     brown    convex      smooth  poisonous            free   
1          bruises    yellow    convex      smooth     edible            free   
2          bruises     white      bell      smooth     edible            free   
3          bruises     white    convex       scaly  poisonous            free   
4               no      gray    convex      smooth     edible            free   
...            ...       ...       ...         ...        ...             ...   
8119            no     brown   knobbed      smooth     edible        attached   
8120            no     brown    convex      smooth     edible        attached   
8121            no     brown      flat      smooth     edible        attached   
8122            no     brown   knobbed       scaly  poisonous            free   
8123            no     brown    convex      smooth     edible        attached   

                                                      ...              \
attribute gill-color gill-size gill-spacing  habitat  ...   ring-type   
record_id                                             ...               
0              black    narrow        close    urban  ...     pendant   
1              black     broad        close  grasses  ...     pendant   
2              brown     broad        close  meadows  ...     pendant   
3              brown    narrow        close    urban  ...     pendant   
4              black     broad      crowded  grasses  ...  evanescent   
...              ...       ...          ...      ...  ...         ...   
8119          yellow     broad        close   leaves  ...     pendant   
8120          yellow     broad        close   leaves  ...     pendant   
8121           brown     broad        close   leaves  ...     pendant   
8122            buff    narrow        close   leaves  ...  evanescent   
8123          yellow     broad        close   leaves  ...     pendant   

                                                                           \
attribute spore-print-color stalk-color-above-ring stalk-color-below-ring   
record_id                                                                   
0                     black                  white                  white   
1                     brown                  white                  white   
2                     brown                  white                  white   
3                     black                  white                  white   
4                     brown                  white                  white   
...                     ...                    ...                    ...   
8119                   buff                 orange                 orange   
8120                   buff                 orange                 orange   
8121                   buff                 orange                 orange   
8122                  white                  white                  white   
8123                 orange                 orange                 orange   

                                                           \
attribute stalk-root stalk-shape stalk-surface-above-ring   
record_id                                                   
0              equal   enlarging                   smooth   
1               club   enlarging                   smooth   
2               club   enlarging                   smooth   
3              equal   enlarging                   smooth   
4              equal    tapering                   smooth   
...              ...         ...                      ...   
8119         missing   enlarging                   smooth   
8120         missing   enlarging                   smooth   
8121         missing   enlarging                   smooth   
8122         missing    tapering          